Recall that the coboundary operator is

$$\partial: C^k(\mathfrak{m},\mathfrak{g})\to C^{k+1}(\mathfrak{m},\mathfrak{g})$$

and is defined by

$$\partial\phi(\alpha_0,\ldots,\alpha_{k}) = \sum_{i=0}^{k}(-1)^i\big[\alpha_i,\phi(\alpha_0,\ldots,\hat\alpha_i,\ldots, \alpha_{k})\big]$$
$$+ \sum_{i<j}(-1)^{i+j}\phi\big([\alpha_i,\alpha_j],\alpha_0,\ldots, \hat \alpha_i,\ldots,\hat\alpha_j,\ldots,\alpha_{k}\big).$$

For a basis element $\phi = X_1^*\wedge\cdots\wedge X_k^*\otimes Y$, we have
$$
    \partial\phi = \sum_{a\not\in\{1,\ldots,k\}} X_a^*\wedge X_1^*\wedge\cdots \wedge X_k^*\otimes[X_a,Y]
    \\
    +\sum_{a<b\not\in\{1,\ldots, k\}}\sum_{i\in \{1,\ldots, k\}}\Big(-X_i^* [X_a,X_i]\Big)X_a^*\wedge X_b^*\wedge X_1^*\wedge\cdots\wedge\widehat {X_i^*}\wedge\cdots X_k^*\otimes Y
$$


In [ ]:
%run Distribution_of_constant_symbol.ipynb

We consider the inner product on the Tanaka symbol with orthonormal basis $(Y,H,E,X,\varepsilon_1,\ldots,\varepsilon_{2m},\eta)$ and lengths

$$|Y|^2=|X|^2=1,|H|^2=|E|^2=2,|\varepsilon_i|^2=\frac{(i-1)!}{(2m-i)!}, |\eta|^2=1$$

along with the innerproduct induced on tensor spaces. In particular, 

$$|A^*\wedge B^*\otimes C|^2 = \frac{|C|^2}{|A|^2|B|^2}$$

In [1]:
%run T_symb.ipynb
%run helpers.ipynb
import copy
from sympy import *
from itertools import combinations
import time
from sympy.matrices.sparsetools import _doktocsr

In [2]:
def key_from_neg(c_key):
    '''arg: c_key a tuple of strings like ('H','X','e1')
       returns: True if c_key is from the complex C(m,g), False otherwise.'''
    # # To do: Rewrite this to be more general
    for i in range(len(c_key)-1):
        A=c_key[i]
        if A in ('H','E','Y'): return False
    return True

In [3]:
class Tensor_alg(object):
    def __init__(self,T_symb_obj):
        self.alg=T_symb_obj
        self.basis_cache={}
        self.dwi_dicts={} # Keys: str_reps of basis elts; Values:(deg,wght,index)
        self.childcls=None
        
    def elt(self,vd):
        return self.childcls(self,vd)

    def elt_from_cd(self,cd={}):
        return self.childcls.from_cd(self,cd)
    
    def basis(self,deg,wght=None):
        """Returns a basis for (deg, wght) as a list, or if wght==None, returns 
        basis for deg as a dict of wghts."""
        if deg<0: return []
        self.init_basis(deg)
        if wght==None:
            return self.basis_cache[deg]
        if deg not in self.basis_cache or wght not in self.basis_cache[deg]: return []
        return self.basis_cache[deg][wght]
    
    def basis_strs(self,deg,wght=None):
        b=self.basis(deg,wght)
        if wght==None:
            r={}
            for k in b:
                r[k]=[tuple([str(c) for c in A.components]) for A in b[k]]
            return r
        return [tuple([str(c) for c in A.components]) for A in b]
        
    def tuple_deg(self,t):
        NotImplemented
    
    def tuple_wght(self,t):
        NotImplemented
    
    def sort_tuple(self,t):
        NotImplemented
        
    def cd_to_vd(self,cd={}):
        """Converts a coeff dict to a vector dict in the basis of self
        """
        r={}
        for A in cd:
            d=self.tuple_deg(A)
            w=self.tuple_wght(A)
            A1,s=self.sort_tuple(A)
            if d not in r: r[d]={}
            if w not in r[d]: r[d][w]=SparseMatrix(zeros(len(self.basis(d,w)),1))
            v=SparseMatrix(zeros(len(self.basis(d,w)),1))
            v[self.dwi_dicts[A1][2]]=s*cd[A]
            r[d][w]=r[d][w]+v
        return r
    
    def Q(self,d,w):
        """Returns the inner product matrix induces by that on the 
        base algebra for degree d and weight w
        INPUTS:
        * 'd' - a degree
        * 'w' - a weight
        """
        if self.alg.Q==None: print('Inner product on base algebra not initialized')
        return SparseMatrix(diag(*[A.length for A in self.basis(d,w)]))
    
    def iprod(self,elt1,elt2):
        """Returns the inner product of elt1 and elt2
        INPUTS:
        * 'elt1', 'elt2' - elements of self
        """
        return elt1.iprod(elt2)
    
    

In [4]:
class Tensor_alg_elt(object):
    def __init__(self,parent,vd={}):
        """INPUTS:
        * 'parent' - A tensor algebra
        * 'vd' - a dict of dicts of vectors, keyed by degree then weight"""
        self.parent=parent
        self.vd=vd
    
    @classmethod
    def from_cd(cls,parent,cd={}):
        NotImplemented
    
    def __str__(self):
        return str_from_vd(self.vd,self.parent.basis)
    
    def __repr__(self):
        return self.__str__()
    
    def __eq__(self,other):
        if type(other)==type(None): return False
        z=self-other
        return z.is_zero()
    
    def is_zero(self):
        for d in self.vd:
            for w in self.vd[d]:
                if simplify(self.vd[d][w])!=zeros(*shape(self.vd[d][w])): return False
        return True
    
    def __neg__(self):
        nvd={d:copy.copy(self.vd[d]) for d in self.vd}
        for d in nvd:
            for w in nvd[d]:
                nvd[d][w]=-nvd[d][w]
        return self.parent.elt(nvd)
        
    def __add__(self,other):
        nvd={}
        for d in set(self.vd.keys()).union(other.vd.keys()):
            nvd[d]={}
            if d in self.vd:
                if d in other.vd: 
                    for w in set(self.vd[d].keys()).union(set(other.vd[d].keys())):
                        if w in self.vd[d]:
                            if d not in nvd: nvd[d]={}
                            nvd[d][w]=copy.copy(self.vd[d][w])
                            if w in other.vd[d]:
                                nvd[d][w]=nvd[d][w]+other.vd[d][w]
                        else: nvd[d][w]=other.vd[d][w]
                else: nvd[d]=copy.copy(self.vd[d])
            else: nvd[d]=copy.copy(other.vd[d])
        return self.parent.elt(nvd)
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,k):
        nvd={}
        for d in self.vd:
            nvd[d]={}
            for w in self.vd[d]:
                nvd[d][w]=k*self.vd[d][w]
        return self.parent.elt(nvd)
    
    def __rmul__(self,other):
        return self*other
    
    def clear_zeros(self):
        """clears zeros from the vector dict of self, leaving only necessary keys"""
        
        for d in self.vd:
            zero_keys=[]
            for w in self.vd[d]:
                if self.vd[d][w]==zeros(*self.vd[d][w].shape): zero_keys.append(w)
            for w in zero_keys:self.vd[d].pop(w)
        
        zero_keys=[]
        for d in self.vd:
            if self.vd[d]=={}: zero_keys.append(d)
        for d in zero_keys: self.vd.pop(d)
    
    def iprod(self,other):
        if not hasattr(other,'parent'): raise ValueError('arg of iprod must have parent')
        if self.parent!=other.parent: raise invalid_parent_exception('args of iprod must have the same parent')
        r=0
        for d in set(self.vd.keys()).intersection(set(other.vd.keys())):
            for w in set(self.vd[d].keys()).intersection(set(other.vd[d].keys())):
                r+=(self.vd[d][w].transpose()*self.parent.Q(d,w)*other.vd[d][w])[0]
        return r
    
    def subs(self,subs_dict):
        return self.xreplace(subs_dict)
    
    def xreplace(self,subs_dict):
        nvd={d:copy.copy(self.vd[d]) for d in self.vd}
        for d in nvd:
            for w in nvd[d]:
                nvd[d][w]=nvd[d][w].xreplace(subs_dict)
        return self.parent.elt(nvd)
    
    def large_subs(self,subs_dict):
        # # To Do
        # cd=copy.copy(self.coeff_dict)
        # for k in cd:
        #     IF=Indexed_factors(cd[k])
        #     NS={}
        #     for A in IF:
        #         if A in subs_dict: NS[A]=subs_dict[A]
        #     cd[k]=cd[k].subs(NS)
        # return self.parent.cochain(cd)
        NotImplemented
        
    def update_add(self,d,w,i,c):
        """Adds c times the specified basis element to self
        INPUTS:
        * 'd' = degree
        * 'w' = weight
        * 'i' = index
        * 'c' = coefficient
        """
        if not d in self.vd: self.vd[d]={}
        if not w in self.vd[d]: self.vd[d][w]=SparseMatrix(zeros(len(self.parent.basis(d,w)),1))
        self.vd[d][w][i]=self.vd[d][w][i]+c
        
    def update_add_vec(self,d,w,v):
        """Adds vector v to the specified degree and weight of self
        INPUTS:
        * 'd' - degree
        * 'w' - weight
        * 'v' - vector
        """
        if not d in self.vd: self.vd[d]={}
        if not w in self.vd[d]: self.vd[d][w]=SparseMatrix(v)
        self.vd[d][w]=SparseMatrix(self.vd[d][w]+v)
        
    def update_add_dict(self,ovd):
        """Adds the vector dict ovd to self
        INPUTS:
        * 'ovd' - a vector dict
        """
        for d in ovd:
            for w in ovd[d]: self.update_add_vec(d,w,ovd[d][w])

In [6]:
class cochain_complex(Tensor_alg):
    def __init__(self,T_symb_obj):
        T_symb_obj.cochain_complex=self
        Tensor_alg.__init__(self,T_symb_obj)
        self.alg=T_symb_obj
        self.ext_alg=T_symb_obj.ext_alg
        self.childcls=cochain
        self.cb_mat_cache={}
        self.subspace_cache={}
        self.rnc=None
    
    def tuple_wght(self,t):
        ''' Returns the weight of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        r=0
        for i in range(len(t)-1):
            j=self.alg.basis_strs.index(t[i])
            r=r-self.alg.wght_list[j] # This is the exterior algebra of m_dual
        r+=self.alg.wght_list[self.alg.basis_strs.index(t[-1])]
        return r
        
    def tuple_deg(self,t):
        ''' Returns the degree of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        return len(t)-1
    
    def dwi(self,basis_str_tuple):
        """Returns the degree, weight, and index of basis_tuple
        Inputs:
        * 'basis_tuple' -- a tuple of basis strings representing a basic cochain
        """
        if basis_str_tuple not in self.dwi_dicts: self.init_basis(len(basis_str_tuple)-1)
        return self.dwi_dicts[basis_str_tuple]
    
    def sort_tuple(self,t):
        """Returns a (cochain)sorting of t and the sign of the corresponding permutation, as a tuple
        INPUTS:
        * 't' - a tuple of basis_strs
        """
        temp=t[0:-1]
        if len(temp)!=len(set(temp)): return (None, 0)
        r,s=sort_basis_tuple(temp,self.alg.basis_strs)
        return (tuple(list(r)+[t[-1]]),s)
        
    def init_basis(self,deg):
        """Sets value of deg in basis_cache and adds to basis_dicts"""
        if deg in self.basis_cache: return None
        deg_subsets=[]
        for A in combinations([str(A) for A in self.alg.m_basis],deg):
            for B in self.alg.basis_strs:
                deg_subsets.append(tuple(list(A)+[B]))
        self.basis_cache[deg]={}
        wght_ct={}
        for i in range(len(deg_subsets)):
            # count the number of elements of deg d and wght w
            # and set the deg, wght, and index of each elt in dwi_dicts
            A=deg_subsets[i]
            w=self.tuple_wght(A)
            if not w in wght_ct: 
                j=0
                wght_ct[w]=1
            else: 
                j=wght_ct[w]
                wght_ct[w]+=1
            self.dwi_dicts[A]=(deg,w,j)
        for A in deg_subsets:
            d,w,i=self.dwi_dicts[A]
            vec=SparseMatrix(zeros(wght_ct[w],1))
            vec[i]=1
            vd={d:{w:vec}}
            b_elt=cochain_basis_elt(self,d,w,vd,A)
            if i==0:self.basis_cache[deg][w]=[b_elt]
            else: self.basis_cache[deg][w].append(b_elt)
                
    def cb_mat(self,d,w):
        """Returns the coboundary matrix which operates on C^d_w
        INPUTS:
        * 'd' - degree
        * 'w' - weight
        """
        if d in self.cb_mat_cache and w in self.cb_mat_cache[d]: return self.cb_mat_cache[d][w]
        if self.basis(d+1,w)==[]: self.cb_mat_cache[(d,w)]=zeros(0,len(self.basis(d,w)))
        elif self.basis(d,w)==[]: self.cb_mat_cache[(d,w)]=zeros(len(self.basis(d+1,w)),0)
        else: self.cb_mat_cache[(d,w)]=SparseMatrix([list(c.cb_vec()) for c in self.basis(d,w)]).transpose()
        return self.cb_mat_cache[(d,w)]
    
    def cb(self, c):
        '''Returns the coboundary map of C(m,g) applied to c
        INPUTS:
        * 'c' - a cochain with self as parent'''
        return c.cb()
    
    def subspace_proj(self,c,subspace):
        """Returns the projection of c onto subspace
        INPUTS:
        * 'subspace' - among 'closed', 'coclosed', 'exact', 'coexact', and 'harmonic'
        * 'c' - a cochain from self
        """
        r={}
        for d in c.vd:
            r[d]={}
            for w in c.vd[d]:
                r[d][w]=ortho_proj(c.vd[d][w],self.subspace_basis(subspace,d,w),self.Q(d,w))
        return self.elt(r)
        
    def subspace_basis(self,subspace,d,w):
        """Returns a basis for the subspace in degree d and weight w
        INPUTS:
        * 'subspace' - among 'closed', 'coclosed', 'exact', 'coexact', and 'harmonic'
        * 'd' - a degree
        * 'w' - a weight
        """
        if len(self.basis(d,w))==0: return Matrix([])
        if (subspace,d,w) in self.subspace_cache: return self.subspace_cache[(subspace,d,w)]
        
        # Should I be caching here? It may be a waste of memory...
        if subspace=='closed':
            col_list=self.cb_mat(d,w).nullspace()
        if subspace=='coclosed':
            col_list=Mat_adjoint(self.cb_mat(d-1,w),self.Q(d-1,w),self.Q(d,w)).nullspace()
        if subspace=='exact':
            col_list=self.cb_mat(d-1,w).columnspace()
        if subspace=='coexact':
            col_list=Mat_adjoint(self.cb_mat(d,w),self.Q(d,w),self.Q(d+1,w)).columnspace()
        if subspace=='harmonic':
            N=(self.cb_mat(d,w)*self.subspace_basis('coclosed',d,w)).nullspace()
            col_list=[self.subspace_basis('coclosed',d,w)*v for v in N]
        if len(col_list)==0: self.subspace_cache[(subspace,d,w)]=zeros(len(self.basis(d,w)),0)
        else: self.subspace_cache[(subspace,d,w)]=Matrix([list(A) for A in col_list]).transpose()
        return self.subspace_cache[(subspace,d,w)]  
    
    def cb_preim_elt(self,c):
        """Returns a cochain which maps to c under the coboundary.
        If c is not exact, returns None.
        INPUTS:
        * 'c' - an exact cochain
        """
        r={}
        for d in c.vd:
            for w in c.vd[d]:
                t=new_Mat_preim_elt(self.cb_mat(d-1,w),c.vd[d][w])
                if t==None: return None
                if d-1 not in r:
                    r[d-1]={}
                if w not in r[d-1]: r[d-1][w]=t
                else: r[d-1][w]=r[d-1][w]+t
        return self.elt(r)
    
    def curv_dict_to_cochain(self,c):
        """Returns a cochain from self representing the structure function or curvature c
        INPUTS:
        * 'c' -- a dictionary repping a structure function {(i,j): vec rep of [Xi,Xj]}
        """
        r={}
        for i in range(len(self.alg.basis)-len(self.alg.m_basis),len(self.alg.basis)):
            for j in range(i+1,len(self.alg.basis)):
                for k in range(len(self.alg.basis)):
                    if c[(i,j)][k]!=0:
                        r[(self.alg.basis_strs[i],self.alg.basis_strs[j],self.alg.basis_strs[k])]=c[(i,j)][k]
        return self.elt_from_cd(r)

    def curv_cochain_to_dict(self,c):
        """Returns a dict of form {(i,j): v such that F*v=[Fi,Fj]-[Xi,Xj]} representing the cochain
        * 'c' -- a dictionary repping a structure function {(i,j): vec rep of [Xi,Xj]}
        """
        r={}
        for i in range(len(self.alg.basis)-len(self.alg.m_basis),len(self.alg.basis)):
            for j in range(i+1,len(self.alg.basis)):
                r[(i,j)]=zeros(len(self.alg.basis),1)

        if not (2 in c.vd): return r
        for w in c.vd[2]:
            v=c.vd[2][w]
            for t in range(len(c.vd[2][w])):
                if v[t]!=0:
                    i,j,k=[self.alg.basis.index(A) for A in self.basis(2,w)[t].components]
                    r[(i,j)][k]=v[t]
        return r       
    
    def one_cochain_mat_rep(self,c):
        """Returns a matrix representation of c as an element of Hom(g,g)
        INPUTS:
        * 'c' - a 1 cochain from self
        """
        if 1 not in c.vd: return zeros(len(self.alg.basis))
        r=zeros(len(self.alg.basis))
        for w in c.vd[1]:
            for a in range(len(c.vd[1][w])):
                # To do
                i,j=[self.alg.basis.index(A) for A in self.basis(1,w)[a].components]
                r[j,i]=c.vd[1][w][a]
        return r
    
    def regular_normal_2_cochain(self,str_rep):
        """Constructs a regular and normal (that is, positive and coclosed) 2-cochain
        with string representation str_rep"""
        if self.rnc==None: 
            eta=C.elt({})
            # # Construct a regular normal 2-cochain
            for i in range(3,len(g.basis)):
                for j in range(i+1, len(g.basis)):
                    for k in range(len(g.basis)):
                        if g.basis[k].wght-g.basis[i].wght-g.basis[j].wght>0:
                            t=(g.basis_strs[i],g.basis_strs[j],g.basis_strs[k])
                            eta=eta+C.elt_from_cd({t:IndexedBase('eta')[i,j,k]})

            c_eta=C.subspace_proj(eta,'exact')

            norm_eqs=[]
            for d in c_eta.vd:
                for w in c_eta.vd[d]:
                    for i in range(c_eta.vd[d][w].shape[0]):
                        if c_eta.vd[d][w][i,0]!=0: norm_eqs.append(c_eta.vd[d][w][i,0])

            norm_sols=solve(norm_eqs,check=False,simplify=False)

            eta=eta.subs(norm_sols)
            simplify_cochain(eta)
            self.rnc=eta
        return self.rnc.subs({IndexedBase('eta'):IndexedBase(str_rep)})

The above (right) $GL_+(\mathfrak{g})$ action on $C^2(\mathfrak{m},\mathfrak{g})$ is given by 

$$A.c(u,v) = A^{-1}\cdot c(Au,Av)\quad \text{or}\quad A.(X_1^*\wedge X_2^*\otimes X_3) = (A^TX_1^*)\wedge(A^TX_2^*)\otimes A^{-1}X_3,$$

which is the naturally induced action. Representing $c$ as a matrix $M_c$, we can also write

$$ M_{A. c} = A^{-1}\cdot c\cdot (A\wedge A)|_{\mathfrak{m}\wedge \mathfrak{m}}$$

In [ ]:
class ext_alg(Tensor_alg):
    def __init__(self,T_symb_obj):
        T_symb_obj.ext_alg=self
        Tensor_alg.__init__(self,T_symb_obj)
        self.childcls=ext_elt
    
    def wedge_tuples(self,tuple1,tuple2,obj2_type):
        '''Returns (wedge,sgn), where wedge is a tuple representing 
        tuple1 wedge tuple2 and sgn is -1 or 1
        
        INPUTS:
        * 'tuple1', 'tuple2' - tuples of T_symb_basis_elt objects
        * 'obj2_type' - among 'ext_elt' and 'cochain', indicating the type of 
                        the object repped by tuple2'''
        if obj2_type not in ['ext_elt','cochain']: raise invalid_parent_exception('wedge_tuples recieved invalid parent type as arg')
        if obj2_type == 'ext_elt': B=tuple2
        else: B=tuple(list(tuple2)[0:-1])
        #check for repeats
        if len(set(tuple1).union(set(B)))!=len(tuple1)+len(B):
            return 'Nil'
        t,s=sort_basis_tuple(tuple1+B,self.alg.basis_strs)
        if obj2_type=='ext_elt': return (t,s)
        return (t+(tuple2[-1],),s)
    
    def tuple_wght(self,t):
        ''' Returns the weight of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        r=0
        for A in t:
            i=self.alg.basis_strs.index(A)
            r=r-self.alg.wght_list[i] # This is the exterior algebra of m_dual
        return r
        
    def tuple_deg(self,t):
        ''' Returns the degree of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        return len(t)
    
        
    def dwi(self,basis_str_tuple):
        """Returns the degree, weight, and index of basis_tuple
        Inputs:
        * 'basis_tuple' -- a tuple of basis strings representing a basic cochain
        """
        if basis_str_tuple not in self.dwi_dicts: self.init_basis(len(basis_str_tuple))
        return self.dwi_dicts[basis_str_tuple]
    
    def sort_tuple(self,t):
        """Returns an (ext alg) sorting of t and the sign of the corresponding permutation, as a tuple
        or None if t contains repeats
        INPUTS:
        * 't' - a tuple of basis_strs
        """
        if len(t)!=len(set(t)): return (None,0)
        return(sort_basis_tuple(t,self.alg.basis_strs))
            
    def init_basis(self,deg):
        """Sets value of deg in basis_cache and adds to basis_dicts"""
        if deg in self.basis_cache: return None
        deg_subsets=[A for A in list(combinations([str(A) for A in self.alg.m_basis],deg))]
        self.basis_cache[deg]={}
        wght_ct={}
        for i in range(len(deg_subsets)):
            # count the number of elements of deg d and wght w
            # and set the deg, wght, and index of each elt in dwi_dicts
            A=deg_subsets[i]
            w=self.tuple_wght(A)
            if not w in wght_ct: 
                j=0
                wght_ct[w]=1
            else: 
                j=wght_ct[w]
                wght_ct[w]+=1
            self.dwi_dicts[A]=(deg,w,j)
        for A in deg_subsets:
            d,w,i=self.dwi_dicts[A]
            vec=SparseMatrix(zeros(wght_ct[w],1))
            vec[i]=1
            vd={d:{w:vec}}
            b_elt=ext_basis_elt(self,d,w,vd,A)
            if i==0:self.basis_cache[deg][w]=[b_elt]
            else: self.basis_cache[deg][w].append(b_elt)
            
    def wedge_indices(self,d1,w1,i1,d2,w2,i2,obj2_type):
        """Returns a tuple with deg, wght, index, and sign of the tuples
        or ('Nil','Nil','Nil',0) if the wedge is zero
        INPUTS:
        * 'd1','d2' - degrees
        * 'w1','w2' - weights
        * 'i1','i2' - indices
        * 'obj2_type' - either 'ext_elt' or 'cochain', giving the type of 
                        the object specified by (d2,w2,i2)
        """
        if obj2_type not in ['ext_elt','cochain']: raise invalid_parent_exception('wedge_indices recieved invalid parent as arg')
        
        t1=tuple([str(A) for A in self.basis(d1,w1)[i1].components])
        if obj2_type=='ext_elt': t2=tuple([str(A) for A in self.basis(d2,w2)[i2].components])
        else: t2=tuple([str(A) for A in self.alg.cochain_complex.basis(d2,w2)[i2].components])
        
        r=self.wedge_tuples(t1,t2,obj2_type)
        if r=='Nil': return ('Nil','Nil','Nil',0)
        t3,s=(r[0],r[1]) # resulting tuple and sign
        if obj2_type=='ext_elt':i3=self.basis_strs(d1+d2,w1+w2).index(t3)
        else: i3=self.alg.cochain_complex.basis_strs(d1+d2,w1+w2).index(t3)
        return (d1+d2,w1+w2,i3,s) 

In [ ]:
class cochain(Tensor_alg_elt):
    def __init__(self,parent,vd={}):
        Tensor_alg_elt.__init__(self,parent,vd)
        
    @classmethod
    def from_cd(cls,parent,cd={}):
        """Constructs an exterior element in parent from a coefficient dictionary"""
        return parent.elt(parent.cd_to_vd(cd))
    
    def __gt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)>str(other)
    
    def __ge__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)>=str(other)
    
    def __lt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)<str(other)
    
    def __le__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)<=str(other)

    def symb_expr(self):
        expr=0
        for d in self.vd:
            for w in self.vd[d]:
                for i in range(len(self.vd[d][w])):
                    if self.vd[d][w][i]!=0: expr+=self.vd[d][w][i]*self.parent.basis(d,w)[i].symb
        return expr

    def pprint(self):
        expr=0
        for d in self.vd:
            for w in self.vd[d]:
                for i in range(len(self.vd[d][w])):
                    if self.vd[d][w][i]!=0: expr+=self.vd[d][w][i]*self.parent.basis(d,w)[i].symb
        display(simplify(expr))
    
    def cb(self):
        """Returns the coboundary map of C(m,g) applied to c
           """
        r={}
        for d in self.vd:
            for w in self.vd[d]:
                if shape(self.parent.cb_mat(d,w))[1]==0: v=zeros(len(self.parent.basis(d+1,w)),1)
                else: v=self.parent.cb_mat(d,w)*self.vd[d][w]
                if d+1 not in r: r[d+1]={}
                if w not in r[d+1]: r[d+1][w]=SparseMatrix(zeros(len(self.parent.basis(d+1,w)),1))
                r[d+1][w]=r[d+1][w]+v
        return self.parent.elt(r)
    
    def cb_preim_elt(self):
        """Returns a cochain which maps to self under the coboundary.
        If self is not exact, returns None.
        """
        return self.parent.cb_preim_elt(self)
    
    def check_valid_vd(self):
        """Returns True if self has a vector dictionary which is a 
        valid representation of a cochain from self.parent, False otherwise
        """
        for d in self.vd:
            for w in self.vd[d]:
                if shape(self.vd[d][w])!=(len(self.parent.basis(d,w)),1): return False
        return True

    def wght_proj(self,w):
        """returns a cochain representing the projection of self onto weight w
        INPUTS:
        * 'w' -- an integer weight
        """
        nvd={}
        for d in self.vd:
            if w in self.vd[d]:
                nvd[d]={w:self.vd[d][w]}
        return self.parent.elt(nvd)
    
    def deg_proj(self,d):
        """returns a cochain representing the projection of self onto weight w
        INPUTS:
        * 'd' -- a nonnegative integer degree
        """
        if d not in self.vd: return self.parent.elt({})
        return self.parent.elt({d:self.vd[d]})

    def apply_cochain_map_base(self,*elts):
        """returns self(elt), taking self to be an element of Hom(Lambda^* m,g)
        INPUTS:
        * 'elt_list' - elements from the Lie algebra basis, all of negative weight
        """
        for A in elts:
            if A.wght>-1: print('apply_cochain_map_base only accepts arguments of negative weight')
                
        r=self.parent.alg.elt()
        ord_strs,s=self.parent.alg.ext_alg.sort_tuple(tuple([str(A) for A in elts]))
        if ord_strs==None: return r
        for A in self.parent.alg.basis:
            d,w,i=self.parent.dwi(tuple(list(ord_strs)+[str(A)]))
            if d in self.vd and w in self.vd[d]: r+=s*self.vd[d][w][i]*A
        return r

    def apply_cochain_map(self,ext_elt):
        """returns self(ext_elt), taking self to be an element of Hom(Lambda^* m,g)
        INPUTS:
        * 'ext_elt' -- an element of the exterior algebra of the Lie algebra
        """
        T=self.parent.alg
        E=self.parent.alg.ext_alg
        r=T.elt()
        for d in ext_elt.vd:
            for w in ext_elt.vd[d]:
                for i in range(len(ext_elt.vd[d][w])):
                    if ext_elt.vd[d][w]!=0:
                        b=E.basis(d,w)[i]
                        r+=ext_elt.vd[d][w][i]*self.apply_cochain_map_base(*b.components)
        return r

    def one_cochain_mat_rep(self):
        return self.parent.one_cochain_mat_rep(self)

In [ ]:
class ext_elt(Tensor_alg_elt):
    def __init__(self,parent,vd={}):
        Tensor_alg_elt.__init__(self,parent,vd)
        
    @classmethod
    def from_cd(cls,parent,cd={}):
        """Constructs an exterior element in parent from a coefficient dictionary"""
        return parent.elt(parent.cd_to_vd(cd))

    def wedge(self,other):
        """ Returns the wedge product of self and other
        INPUTS:
        * 'other' - another exterior element or a cochain
        
        NOTE: Since it's ambiguous whether an algebra element is dual or not, 
              other cannot be of type T_symb_elt
        """
        
        obj2_type=None        
        if isinstance(other.parent,ext_alg):
            obj2_type='ext_elt'
            r=self.parent.elt({})
        if isinstance(other.parent,cochain_complex):
            obj2_type='cochain'
            r=self.parent.alg.cochain_complex.elt({})
        if obj2_type==None: raise invalid_parent_exception('wedge recieved arguments with incompatible parents')
        
        L1=[]
        for d1 in self.vd:
            for w1 in self.vd[d1]:
                for i1 in range(len(self.vd[d1][w1])):
                    if self.vd[d1][w1][i1]!=0: L1.append((d1,w1,i1,self.vd[d1][w1][i1]))
        L2=[]
        for d2 in other.vd:
            for w2 in other.vd[d2]:
                for i2 in range(len(other.vd[d2][w2])):
                    if other.vd[d2][w2][i2]!=0: L2.append((d2,w2,i2,other.vd[d2][w2][i2]))
        for t1 in L1:
            d1,w1,i1,c1=t1
            for t2 in L2:
                d2,w2,i2,c2=t2
                d3,w3,i3,s=self.parent.wedge_indices(d1,w1,i1,d2,w2,i2,obj2_type)
                if s!=0: r.update_add(d3,w3,i3,s*c1*c2)
        return r
    
    def tensor(self,v):
        """Returns self otimes v, and element of the Chevalley-Eilenberg Complex
        INPUTS:
        * 'v' - an element of the Lie algebra of self
        """
        r=self.parent.alg.cochain_complex.elt({})
        for d in self.vd:
            for w in self.vd[d]:
                for i in range(len(self.vd[d][w])):
                    if self.vd[d][w][i]!=0:
                        r=r+self.vd[d][w][i]*self.parent.basis(d,w)[i].tensor(v)
        return r

In [ ]:
class Tensor_alg_basis_elt(Tensor_alg_elt):
    def __init__(self,parent,deg,wght,vd,str_rep):
        self.deg=deg
        self.wght=wght
        self.str_rep=str_rep
        Tensor_alg_elt.__init__(self,parent,vd)
        alg=parent.alg
        self.components=[alg.basis[alg.basis_strs.index(A)] for A in str_rep]
    
    def __str__(self):
        return tuple_to_str(self.str_rep)
    
    def __repr__(self):
        return self.__str__()

In [ ]:
class ext_basis_elt(ext_elt,Tensor_alg_basis_elt):
    def __init__(self,parent,deg,wght,vd,str_rep):
        ext_elt.__init__(self,parent,vd)
        Tensor_alg_basis_elt.__init__(self,parent,deg,wght,vd,str_rep)
        self.length=None
        if parent.alg.Q!=None:
            self.length=Rational(1,prod([A.length for A in self.components]))

    def tensor(self,v):
        """Returns self otimes v, and element of the Chevalley-Eilenberg Complex
        INPUTS:
        * 'v' - an element of the Lie algebra of self
        """
        r_dict={}
        for i in range(len(v.vec)):
            if v.vec[i]!=0:
                curr_str=tuple([str(a) for a in self.components]+[self.parent.alg.basis_strs[i]])
                r_dict[curr_str]=v.vec[i]
        return self.parent.alg.cochain_complex.elt_from_cd(r_dict)

For a basis element $\phi = X_1^*\wedge\cdots\wedge X_k^*\otimes Y$, we have
$$
    \partial\phi = \sum_{a\not\in\{1,\ldots,k\}} X_a^*\wedge X_1^*\wedge\cdots \wedge X_k^*\otimes[X_a,Y]
    \\
    +\sum_{a<b\not\in\{1,\ldots, k\}}\sum_{i\in \{1,\ldots, k\}}(-1)^i\Big(X_i^* [X_a,X_b]\Big)X_a^*\wedge X_b^*\wedge X_1^*\wedge\cdots\wedge\widehat {X_i^*}\wedge\cdots X_k^*\otimes Y
$$

In [ ]:
class cochain_basis_elt(cochain,Tensor_alg_basis_elt):
    def __init__(self,parent,deg,wght,vd,str_rep):
        cochain.__init__(self,parent,vd)
        Tensor_alg_basis_elt.__init__(self,parent,deg,wght,vd,str_rep)
        self.length=None
        if parent.alg.Q!=None:
            self.length=Rational(self.components[-1].length,
                                 prod([A.length for A in self.components[0:-1]]))
            
        # Set the symbol representation of self
        sl=[str(A) for A in self.components]
        s='{'*(len(sl)-1)
        for i in range(len(sl)-2):
            s+=sl[i]
            s+='^*\\wedge}'
        s+=sl[len(sl)-2]+'^*\\otimes}'+sl[-1]
        self.symb=symbols(s)
            
    def cb_vec(self):
        """Returns the vector representing coboundary(self) in the basis C^{deg+1}_{wght}(m,g)
        Should only be called by cochain.cb_mat.
        """
        mb=self.parent.alg.m_basis
        Y=self.components[-1]
        r=[0]*len(self.parent.basis(self.deg+1,self.wght))
        
        B_set=set([mb.index(A) for A in self.components[0:-1]])
        non_B_set=set(range(len(mb))).difference(B_set)
        
        # First term
        for a in non_B_set:
            Xa=mb[a]
            # compute the RHS above, im_a_vec: [X_a,Y]+sum_{i} X_i^*[X_a,X_i]Y
            im_a_vec=Xa.ad(Y).vec
            for j in range(len(im_a_vec)):
                if im_a_vec[j]!=0:
                    tl=[Xa]+self.components[0:-1]+[self.parent.alg.basis[j]]
                    t,s=self.parent.sort_tuple(tuple([str(A) for A in tl]))
                    if s!=0:
                        ind=self.parent.basis_strs(self.deg+1,self.wght).index(t)
                        r[ind]+=s*im_a_vec[j]
        # Second term
        for a in non_B_set:
            for b in non_B_set:
                if a<b:
                    Xa=mb[a]
                    Xab=Xa.ad_mat(mod='m').col(b)
                    for i in B_set:
                        if Xab[i]!=0:
                            cpts=copy.copy(self.components[0:-1])
                            cpts.remove(mb[i])
                            tl=[Xa,mb[b]]+cpts+[self.components[-1]]
                            t,s=self.parent.sort_tuple(tuple([str(A) for A in tl]))
                            if s!=0:
                                ind=self.parent.basis_strs(self.deg+1,self.wght).index(t)
                                r[ind]+=(-1)**(1+self.components.index(mb[i]))*s*Xab[i]
        
        return SparseMatrix([r]).transpose()

In [ ]:
class cochain_init_Exception(Exception):
     def __init__(self, message=""):
        self.message = message
        super().__init__(self.message)

In [ ]:
class nonhomogeneous_Exception(Exception):
    def __init__(self,message=""):
        self.message = message
        super().__init__(self.message)

If the curvature $K_{<d}$ is known, then we compute $K_d$ as follows:

$$K_{\geq d}(v_1,v_1) = F^{-1}\Big([F(v_1),F(v_2)]-F[v_1,v_2]-F\circ K_{< d}(v_1,v_2)\Big)$$

## Gerstenhaber Product

In [ ]:
def Apply_Gerst_prod_on_base(f,h,w):
    """returns f circ h (w)
    args:
      * 'f', 'h' - homogeneous elements from a cochain complex
      * 'w' - a basis element of the exterior algebra"""
    if f==f.parent.elt({}) or h==h.parent.elt({}) or w==w.parent.elt({}): return f.parent.alg.elt()
    fd=list(f.vd.keys())[0]
    hd=list(h.vd.keys())[0]
    
    if w.deg!=fd+hd-1: return w.alg.elt([0]*len(w.alg.basis))
    r=0
    for comb in list(combinations(w.components,hd)):
        l1=list(comb)
        l2=[a for a in w.components if a not in comb]
        s=permutation_sign(l1+l2,w.components)

        if len(l1)==0: w1=f.parent.alg.ext_alg.basis(0,0)[0]
        else: w1=l1[0].cast_as_ext_elt()
        if len(l2)==0: w2=f.parent.alg.ext_alg.basis(0,0)[0]
        else: w2=l2[0].cast_as_ext_elt()

        for t in l1[1:len(l1)]:
            w1=w1.wedge(t.cast_as_ext_elt())
        for t in l2[1:len(l2)]:
            w2=w2.wedge(t.cast_as_ext_elt())
        t=f.apply_cochain_map(h.apply_cochain_map(w1).negative_projection().cast_as_ext_elt().wedge(w2))
        r=r+s*t
    return r

In [ ]:
def homog_Gerst_prod(f,h):
    """returns f circ h
    args:
      * 'f', 'h' - deg homogeneous elements from a cochain complex
    """
    if f==f.parent.elt({}) or h==h.parent.elt({}): return f.parent.alg.elt()
    fd=list(f.vd.keys())[0]
    hd=list(h.vd.keys())[0]

    r=f.parent.elt({})
    for w in f.parent.alg.ext_alg.basis(fd+hd-1):
        for e_elt in f.parent.alg.ext_alg.basis(fd+hd-1)[w]:
            t=Apply_Gerst_prod_on_base(f,h,e_elt)
            for i in range(len(t.vec)):
                if t.vec[i]!=0:
                  s=tuple([str(A) for A in e_elt.components]+[t.parent.basis_strs[i]])
                  r+=f.parent.elt_from_cd({s:t.vec[i]})
    return r

def homog_Gerst_bracket(f,h):
    if f==f.parent.elt({}) or h==h.parent.elt({}): return f.parent.alg.elt()
    fd=list(f.vd.keys())[0]
    hd=list(h.vd.keys())[0]

    r=f.parent.elt({})
    for w in f.parent.alg.ext_alg.basis(fd+hd-1):
        for e_elt in f.parent.alg.ext_alg.basis(fd+hd-1)[w]:
            t=Apply_Gerst_prod_on_base(f,h,e_elt)-(-1)**(fd+hd)*Apply_Gerst_prod_on_base(h,f,e_elt)
            for i in range(len(t.vec)):
                if t.vec[i]!=0:
                  s=tuple([str(A) for A in e_elt.components]+[t.parent.basis_strs[i]])
                  r+=f.parent.elt_from_cd({s:t.vec[i]})
    return r

In [ ]:
def Gerst_prod(f,h):
    """returns f circ h"""
    C=f.parent
    r=C.elt({})

    f_degs=list(f.vd.keys())
    h_degs=list(h.vd.keys())

    for fd in f_degs:
        fp=f.deg_proj(fd)
        for hd in h_degs:
            hp=h.deg_proj(hd)
            r=r+homog_Gerst_prod(fp,hp)
    return r

def Gerst_bracket(f,h):
    C=f.parent
    r=C.elt({})

    f_degs=list(f.vd.keys())
    h_degs=list(h.vd.keys())

    for fd in f_degs:
        fp=f.deg_proj(fd)
        for hd in h_degs:
            hp=h.deg_proj(hd)
            r=r+homog_Gerst_bracket(fp,hp)
    return r
